# 06. Madden-Julian Oscillation (MJO) RMM Index Download & Preprocessing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/06_download_mjo_rmm_index.ipynb)

This notebook downloads daily Real-time Multivariate MJO (`RMM1`, `RMM2`)
indices published by the Australian Bureau of Meteorology (Wheeler & Hendon,
2004), masks out the last calendar day of each month to respect the ~2-day
operational data latency on 1st-of-month forecast issuances, and writes
monthly subseasonal teleconnection features to Zarr:
- **Output path**: `${FLOWRACLE_DATA_DIR}/mjo-rmm-index/mjo_rmm_index.zarr`
- **Computed monthly features**: `eom_rmm1`, `eom_rmm2` (mean over last 7
  usable days of the month), `monthly_mjo_amp`, and `active_mjo_days`.

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0"

In [ ]:
import os
import shutil
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import requests
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "mjo-rmm-index")
os.makedirs(OUTPUT_DIR, exist_ok=True)

BOM_RMM_URL = "http://www.bom.gov.au/clim_data/IDCKGEM000/rmm.74toRealtime.txt"
HEADER_LINES = 2
EOM_WINDOW_DAYS = 7
ACTIVE_MJO_THRESHOLD = 1.0

In [ ]:
def download_rmm_daily(url: str = BOM_RMM_URL) -> pd.DataFrame:
  try:
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=60) as resp:
      raw_text = resp.read().decode("utf-8")
  except Exception:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    resp.raise_for_status()
    raw_text = resp.text

  records = []
  for line in raw_text.strip().splitlines()[HEADER_LINES:]:
    parts = line.split()
    if len(parts) < 7:
      continue
    try:
      year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
      rmm1, rmm2 = float(parts[3]), float(parts[4])
      phase, amp = int(parts[5]), float(parts[6])
      date = pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
      continue
    if any(abs(v) > 1e30 for v in (rmm1, rmm2, amp)) or phase == 999:
      rmm1 = rmm2 = phase = amp = np.nan
    records.append({
        "date": date,
        "RMM1": rmm1,
        "RMM2": rmm2,
        "phase": phase,
        "amplitude": amp,
    })

  df = pd.DataFrame(records).set_index("date").sort_index()
  df["amplitude"] = np.sqrt(df["RMM1"] ** 2 + df["RMM2"] ** 2)
  return df


def compute_operational_monthly_mjo(df_daily: pd.DataFrame) -> pd.DataFrame:
  # Exclude the final calendar day of each month (~2-day operational BoM latency).
  is_last_day = df_daily.index == (df_daily.index + pd.offsets.MonthEnd(0))
  df_masked = df_daily[~is_last_day].copy()
  groups = df_masked.groupby(pd.Grouper(freq="1MS"))

  def _eom_tail_mean(group, col):
    valid = group[col].dropna()
    return np.nan if valid.empty else float(valid.tail(EOM_WINDOW_DAYS).mean())

  monthly_df = pd.DataFrame({
      "eom_rmm1": groups.apply(lambda g: _eom_tail_mean(g, "RMM1")),
      "eom_rmm2": groups.apply(lambda g: _eom_tail_mean(g, "RMM2")),
      "monthly_mjo_amp": groups["amplitude"].mean(),
      "active_mjo_days": (
          groups["amplitude"].apply(
              lambda s: float((s > ACTIVE_MJO_THRESHOLD).sum())
          )
      ),
  })
  monthly_df.index.name = "Date"
  return monthly_df

In [ ]:
df_daily = download_rmm_daily()
monthly_df = compute_operational_monthly_mjo(df_daily)

time_coord = monthly_df.index.values
ds = xr.Dataset({
    col: xr.DataArray(
        monthly_df[col].values.astype(np.float64),
        dims=["time"],
        coords={"time": time_coord},
    )
    for col in monthly_df.columns
})
ds.attrs = {
    "Description": (
        "Monthly MJO RMM index features for FNF forecasting. Last day of each "
        "month is excluded to account for ~2-day operational latency."
    ),
    "source": BOM_RMM_URL,
    "reference": "Wheeler & Hendon (2004), Gottschalck et al. (2010)",
    "operational_lag_days": 2,
    "eom_window_days": EOM_WINDOW_DAYS,
    "active_mjo_threshold": ACTIVE_MJO_THRESHOLD,
}

zarr_path = os.path.join(OUTPUT_DIR, "mjo_rmm_index.zarr")
if os.path.exists(zarr_path):
  shutil.rmtree(zarr_path)
ds.to_zarr(zarr_path, mode="w")
print(f"Saved Zarr store: {zarr_path}")
print(ds)